# Shared KofamScan KO per gene

Which KEGG KO does one shared KofamScan run assign to each gene called as an AMG at any tier by any tool, and which Figure 3 class does each gene fall in? The three tools annotate with different databases and versions, so a per-tool KO cannot place their calls on one functional scale. This notebook reads one KofamScan 1.3.0 run over all genes called at any tier (set b), assigns each gene a KO under the primary rule (best hit at an e-value of at most 1e-5, regardless of the adaptive threshold) and under the sensitivity rule (the highest-scoring hit above the KofamScan adaptive threshold, marked `*`), and labels every gene with the four Figure 3 classes and the Class I/II classes of Supplemental Table 9 and Source Data under each rule.

## Inputs

The KofamScan run was made separately and is adopted as an md5-pinned input. Its query FASTA, detail output, `ko_list`, md5 and version files and the timed command were copied without change to `kofamscan_shared_v19` under the benchmark directory, and the cell below refuses a file whose md5 differs from the pinned value. The KO hierarchy is the pinned `./data/ko00001.keg`. The gene table is `amg_predictions_combined.parquet` of the mm13 AMG tables directory, written by `amg_benchmark_data.ipynb`, restricted below to genes called at any tier, the gene set of `AMG_predictions_gene_level.parquet`. It is read upstream of `amg_benchmark_comparisons.ipynb` because that notebook reads this one's output.

In [ ]:
import hashlib
import os
import re
from pathlib import Path

os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl

ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/benchmarks")
GENOME_DIR = ROOT_DIR.joinpath("benchmark_genomes")
MAIN_DIR = ROOT_DIR.joinpath("metabolism_benchmark")
KOFAM_DIR = MAIN_DIR.joinpath("kofamscan_shared_v19")
DETAIL = KOFAM_DIR.joinpath("kofam_set_b.detail.tsv")
SET_B_FAA = KOFAM_DIR.joinpath("set_b.faa")
KO_LIST = KOFAM_DIR.joinpath("ko_list")
KO00001 = Path("./data/ko00001.keg")
GENE_TABLE = MAIN_DIR.joinpath("amg_tables_mm13").joinpath("amg_predictions_combined.parquet")
OUTPUT_TABLES_DIR = Path("./tables/amg_benchmark")
PINNED_MD5 = {DETAIL: "0eff0fffb4dfb10f13799e90bbc2d224", SET_B_FAA: "f191e3765fe0376cbda8ee20d1ef139a",
              KO_LIST: "91075c124ca00c67832435af21ea7a6b"}
INPUTS = [DETAIL, SET_B_FAA, KO_LIST, KO00001, GENE_TABLE]


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()


for path, expected in PINNED_MD5.items():
    assert md5(path) == expected, f"{path.name} does not match its pinned md5"

## Method

### The KofamScan run

KofamScan 1.3.0 (`exec_annotation`) with HMMER 3.4 ran in a scratch conda environment, which is not a project environment, against the KOfam profiles of `profiles.tar.gz` (md5 457ee34873cfbbbfe73418731eb84d4b, not copied, distributed at ftp://ftp.genome.jp/pub/db/kofam/) and `ko_list` (md5 91075c124ca00c67832435af21ea7a6b). The command, recorded in `time_set_b.txt`, was `exec_annotation -p profiles -k ko_list --cpu 16 -f detail-tsv -o kofam_set_b.detail.tsv set_b.faa`, and it exited 0. `set_b.faa` holds one protein per gene of `AMG_predictions_gene_level.parquet`, named `sample|gene` and written by `make_set_b.py` from `dramv_genes_reformatted`.

### KO hierarchy

`ko00001.keg` is parsed into one row per KO and hierarchy position, with its A, B and C nodes and whether the C node is a pathway map (`[PATH:`). A KO is a KEGG Metabolism KO when it sits under A 09100. A KO is Class I when it sits on a pathway map under 09100, and Class II when it sits under B 09191 (Unclassified: metabolism) or B 09131 (Membrane transport) and on no 09100 pathway map.

In [ ]:
membership_rows = []
a_node = b_node = c_node = None
for line in open(KO00001, encoding="utf-8", errors="replace"):
    text = line.rstrip("\n")
    if text.startswith("A"):
        a_node = text[1:].strip()
    elif text.startswith("B") and text[1:].strip():
        b_node = text[1:].strip()
    elif text.startswith("C"):
        c_node = text[1:].strip()
    elif text.startswith("D"):
        parts = text.split(None, 2)
        if len(parts) >= 2 and re.fullmatch(r"K\d{5}", parts[1]):
            membership_rows.append({"ko": parts[1], "ko_name": parts[2] if len(parts) > 2 else "", "A": a_node, "B": b_node,
                                    "C": c_node, "on_map": "[PATH:" in (c_node or "")})
membership = pl.DataFrame(membership_rows).with_columns(pl.col("A").str.slice(0, 5).alias("A_code"),
                                                        pl.col("B").str.slice(0, 5).alias("B_code"))
KOS_ANY = set(membership["ko"])
metabolism = membership.filter(pl.col("A_code") == "09100")
KOS_09100 = set(metabolism["ko"])
KOS_09100_MAP = set(metabolism.filter(pl.col("on_map"))["ko"])
CLASS2_B_NODES = ["09131", "09191"]
KOS_CLASS2 = set(membership.filter(pl.col("B_code").is_in(CLASS2_B_NODES))["ko"]) - KOS_09100_MAP
ko_b_nodes = membership.group_by("ko").agg(pl.col("B").unique().sort().str.join("; ").alias("brite_B"))
ko_list_kos = set(pl.read_csv(KO_LIST, separator="\t", infer_schema_length=0, quote_char=None)["knum"])
print("ko00001 KOs", len(KOS_ANY), "| under 09100", len(KOS_09100), "| on a 09100 map", len(KOS_09100_MAP),
      "| Class II KOs", len(KOS_CLASS2))
print("ko_list KOs not in ko00001", len(ko_list_kos - KOS_ANY), "| ko00001 KOs not in ko_list", len(KOS_ANY - ko_list_kos))

ko00001 holds 28,008 KOs, 6,490 of them under 09100 and 6,489 of those on a 09100 pathway map, so the KEGG Metabolism and Class I KO sets differ by one KO. 2,456 KOs are Class II. 472 KOs of the KOfam `ko_list` are not placed in ko00001, and 27 ko00001 KOs have no KOfam profile.

### KO per gene under each rule

Each detail row is one gene and KO profile with the adaptive threshold, the score, the e-value and a `*` when the score reaches the threshold. Under the primary rule a gene takes the KO of its hit with the lowest e-value among hits at an e-value of at most 1e-5, ties broken by higher score and then KO, whether or not the hit reaches the threshold (`S_ko`). Under the sensitivity rule a gene takes the `*` hit with the highest ratio of score to threshold, ties broken by lower e-value and then KO (`P_ko`), and every `*` hit of the gene is kept in `P_kos`. `P` prefixes the threshold rule and `S` the e-value rule.

In [ ]:
E_VALUE_MAX = 1e-5
DETAIL_COLUMNS = ["sig", "query", "ko", "threshold", "score", "evalue", "definition"]
hits = pl.read_csv(DETAIL, separator="\t", has_header=False, comment_prefix="#", quote_char=None, new_columns=DETAIL_COLUMNS,
                   schema_overrides={"sig": pl.Utf8, "query": pl.Utf8, "ko": pl.Utf8, "threshold": pl.Utf8, "score": pl.Float64,
                                     "evalue": pl.Float64, "definition": pl.Utf8})
hits = hits.with_columns(
    (pl.col("sig").fill_null("").str.strip_chars() == "*").alias("sig"),
    pl.when(pl.col("threshold") == "-").then(None).otherwise(pl.col("threshold")).cast(pl.Float64).alias("threshold"),
    pl.col("query").str.split_exact("|", 1).struct.rename_fields(["sample", "gene"]).alias("q"),
).unnest("q").drop("query", "definition")
print("detail rows", hits.height, "| above threshold", int(hits["sig"].sum()), "| genes with any hit", hits.select("sample", "gene").n_unique())

threshold_rule = (
    hits.filter(pl.col("sig")).with_columns((pl.col("score") / pl.col("threshold")).alias("ratio"))
    .sort(["sample", "gene", "ratio", "evalue", "ko"], descending=[False, False, True, False, False])
    .group_by(["sample", "gene"], maintain_order=True)
    .agg(pl.col("ko").first().alias("P_ko"), pl.col("ratio").first().alias("P_score_over_threshold"), pl.col("ko").alias("P_kos"))
)
evalue_rule = (
    hits.filter(pl.col("evalue") <= E_VALUE_MAX)
    .sort(["sample", "gene", "evalue", "score", "ko"], descending=[False, False, False, True, False])
    .group_by(["sample", "gene"], maintain_order=True)
    .agg(pl.col("ko").first().alias("S_ko"), pl.col("evalue").first().alias("S_evalue"), pl.col("score").first().alias("S_score"))
)
print("genes with a KO: primary rule (S)", evalue_rule.height, "| sensitivity rule (P)", threshold_rule.height)

The run reports 13,279,705 gene and profile rows, 213,931 of them above the adaptive threshold, for 304,899 genes with any hit. The primary rule gives a KO to 299,887 genes and the sensitivity rule to 206,096.

### Classes per gene

The genes are those called at any tier by any tool in the gene table, checked to be exactly the `sample|gene` names of `set_b.faa`. Under the primary rule the four-class label (`S_S2_fill`) is `not geNomad viral` for a gene off a geNomad-viral region, and otherwise `viral, KEGG Metabolism KO`, `viral, KO outside Metabolism` or `viral, no KO` from `S_ko`. The Class I/II label (`S_class12_fill`) splits the viral genes with a KO into Class I, Class II and other KO. Under the sensitivity rule (`P_S2_fill`, `P_class12_fill`) a gene counts as a Metabolism, Class I or Class II gene when any of its `*` hits is one,. A KO that ko00001 does not place counts as a KO outside Metabolism. `S_brite_B` and `P_brite_B` list the BRITE B nodes of `S_ko` and `P_ko`. `DRAMV_aux1` to `DRAMV_aux3` split DRAM-V (default) by auxiliary score. The table is written to `tables/amg_benchmark/shared_ko_per_gene.parquet` beside the other benchmark tables that `amg_benchmark_comparisons.ipynb` and `amg_benchmark_figures.Rmd` read.

In [ ]:
TIERS = ["CheckAMG_high", "CheckAMG_medium", "CheckAMG_low", "DRAMV_default", "DRAMV_allow_T", "DRAMV_aux_4",
         "DRAMV_aux_4_allow_T", "VIBRANT_high", "VIBRANT_medium", "VIBRANT_low"]
genes = (
    pl.read_parquet(GENE_TABLE, columns=["sample", "gene", "source", "ecosystem", "genomad_viral", "DRAMV_auxiliary_score"] + TIERS)
    .filter(pl.any_horizontal(TIERS))
    .with_columns([pl.col(tier).fill_null(False) for tier in TIERS] + [pl.col("genomad_viral").fill_null(False)])
    .with_columns([(pl.col("DRAMV_default") & (pl.col("DRAMV_auxiliary_score") == score)).fill_null(False).alias(f"DRAMV_aux{score}")
                   for score in (1, 2, 3)])
)
assert genes.select("sample", "gene").is_unique().all(), "sample and gene do not identify a gene"
assert (genes["DRAMV_aux1"].cast(pl.Int8) + genes["DRAMV_aux2"].cast(pl.Int8) + genes["DRAMV_aux3"].cast(pl.Int8)
        == genes["DRAMV_default"].cast(pl.Int8)).all(), "the auxiliary-score split is not a partition of DRAM-V (default)"
faa_names = set()
with open(SET_B_FAA) as handle:
    for line in handle:
        if line.startswith(">"):
            faa_names.add(line[1:].strip())
assert faa_names == set(genes["sample"] + "|" + genes["gene"]), "set_b.faa and the genes called at any tier differ"


def brite_class(ko):
    if ko is None:
        return "no_KO"
    if ko not in KOS_ANY:
        return "KO_not_in_ko00001"
    return "metabolism_09100" if ko in KOS_09100 else "other_KO"


def brite_class_any(kos):
    if kos is None or len(kos) == 0:
        return "no_KO"
    if any(ko in KOS_09100 for ko in kos):
        return "metabolism_09100"
    if all(ko not in KOS_ANY for ko in kos):
        return "KO_not_in_ko00001"
    return "other_KO"


def class12(kos):
    if kos is None or len(kos) == 0:
        return "no_KO"
    if any(ko in KOS_09100_MAP for ko in kos):
        return "class_I"
    if any(ko in KOS_CLASS2 for ko in kos):
        return "class_II"
    return "other_KO"


def four_class(viral, brite):
    if not viral:
        return "not geNomad viral"
    if brite == "metabolism_09100":
        return "viral, KEGG Metabolism KO"
    if brite == "no_KO":
        return "viral, no KO"
    return "viral, KO outside Metabolism"


def class12_fill(viral, label):
    if not viral:
        return "not geNomad viral"
    return {"class_I": "viral, Class I", "class_II": "viral, Class II", "other_KO": "viral, other KO", "no_KO": "viral, no KO"}[label]


shared_ko = (
    genes.join(threshold_rule, on=["sample", "gene"], how="left").join(evalue_rule, on=["sample", "gene"], how="left")
    .with_columns(
        pl.col("P_kos").map_elements(brite_class_any, return_dtype=pl.Utf8).fill_null("no_KO").alias("P_brite"),
        pl.col("S_ko").map_elements(brite_class, return_dtype=pl.Utf8).fill_null("no_KO").alias("S_brite"),
        pl.col("P_kos").map_elements(class12, return_dtype=pl.Utf8).fill_null("no_KO").alias("P_class12"),
        pl.col("S_ko").map_elements(lambda ko: class12([ko]), return_dtype=pl.Utf8).fill_null("no_KO").alias("S_class12"),
    )
)
for rule in ["S", "P"]:
    shared_ko = shared_ko.with_columns(
        pl.struct("genomad_viral", f"{rule}_brite").map_elements(lambda row, r=rule: four_class(row["genomad_viral"], row[f"{r}_brite"]),
                                                                 return_dtype=pl.Utf8).alias(f"{rule}_S2_fill"),
        pl.struct("genomad_viral", f"{rule}_class12").map_elements(lambda row, r=rule: class12_fill(row["genomad_viral"], row[f"{r}_class12"]),
                                                                   return_dtype=pl.Utf8).alias(f"{rule}_class12_fill"),
    ).join(ko_b_nodes.rename({"ko": f"{rule}_ko", "brite_B": f"{rule}_brite_B"}), on=f"{rule}_ko", how="left")
shared_ko = shared_ko.select(
    "sample", "gene", "source", "ecosystem", "genomad_viral", "DRAMV_auxiliary_score", *TIERS, "DRAMV_aux1", "DRAMV_aux2", "DRAMV_aux3",
    "S_ko", "S_evalue", "S_score", "S_brite", "S_brite_B", "S_S2_fill", "S_class12", "S_class12_fill",
    "P_ko", "P_kos", "P_score_over_threshold", "P_brite", "P_brite_B", "P_S2_fill", "P_class12", "P_class12_fill",
).sort("sample", "gene")
assert shared_ko.height == genes.height
SHARED_KO_PATH = OUTPUT_TABLES_DIR.joinpath("shared_ko_per_gene.parquet")
shared_ko.write_parquet(SHARED_KO_PATH)
print("genes", shared_ko.height, "| with a KO under the primary rule (S)", shared_ko["S_ko"].is_not_null().sum(),
      "| under the sensitivity rule (P)", shared_ko["P_ko"].is_not_null().sum())
print("S_brite", shared_ko["S_brite"].value_counts().sort("S_brite").rows())
print("P_brite", shared_ko["P_brite"].value_counts().sort("P_brite").rows())

All 306,534 genes called at any tier are in the table, and they are exactly the sequences of `set_b.faa`. Under the primary rule 175,590 genes take a KEGG Metabolism KO, 123,093 a KO outside Metabolism, 1,204 a KO that ko00001 does not place and 6,647 no KO. Under the sensitivity rule the counts are 121,702, 83,961, 433 and 100,438.

### Classes per matched level

The four-class and Class I/II counts of each tool level under both rules, for the nine matched levels, which are cumulative and carry the names of the Figure 3 sets: level 1 is CheckAMG (high), DRAM-V (aux 1) and VIBRANT (high), level 2 is CheckAMG (medium), DRAM-V (aux 2) and VIBRANT (medium), and level 3 is CheckAMG (low), DRAM-V (aux 3) and VIBRANT (low). A CheckAMG or VIBRANT level includes the tiers above it, DRAM-V (aux 2) and (aux 3) hold the calls at auxiliary score at most 2 and at most 3, and DRAM-V (aux 3) is DRAM-V (default). `amg_benchmark_figures.Rmd` writes the Source Data counts from `shared_ko_per_gene` itself.

In [ ]:
LEVELS = {
    "CheckAMG (high)": pl.col("CheckAMG_high"),
    "CheckAMG (medium)": pl.col("CheckAMG_high") | pl.col("CheckAMG_medium"),
    "CheckAMG (low)": pl.col("CheckAMG_high") | pl.col("CheckAMG_medium") | pl.col("CheckAMG_low"),
    "DRAM-V (aux 1)": pl.col("DRAMV_aux1"),
    "DRAM-V (aux 2)": pl.col("DRAMV_aux1") | pl.col("DRAMV_aux2"),
    "DRAM-V (aux 3)": pl.col("DRAMV_default"),
    "VIBRANT (high)": pl.col("VIBRANT_high"),
    "VIBRANT (medium)": pl.col("VIBRANT_high") | pl.col("VIBRANT_medium"),
    "VIBRANT (low)": pl.col("VIBRANT_high") | pl.col("VIBRANT_medium") | pl.col("VIBRANT_low"),
}
MATCHED = list(LEVELS)
FOUR_CLASSES = ["viral, KEGG Metabolism KO", "viral, KO outside Metabolism", "viral, no KO", "not geNomad viral"]
CLASS12_CLASSES = ["viral, Class I", "viral, Class II", "viral, other KO", "viral, no KO", "not geNomad viral"]
RULES = {"S": "primary, best hit at e-value <= 1e-5", "P": "sensitivity, KofamScan adaptive threshold"}
level_rows = []
for level, keep in LEVELS.items():
    calls = shared_ko.filter(keep)
    for rule, rule_label in RULES.items():
        row = {"level": level, "matched_level": level in MATCHED, "rule": rule, "rule_label": rule_label, "n_calls": calls.height,
               "n_genomad_viral": int(calls["genomad_viral"].sum()), "n_shared_ko": int(calls[f"{rule}_ko"].is_not_null().sum())}
        row.update({f"four_class: {label}": int((calls[f"{rule}_S2_fill"] == label).sum()) for label in FOUR_CLASSES})
        row.update({f"class12: {label}": int((calls[f"{rule}_class12_fill"] == label).sum()) for label in CLASS12_CLASSES})
        level_rows.append(row)
level_counts = pl.DataFrame(level_rows)
LEVEL_COUNTS_PATH = OUTPUT_TABLES_DIR.joinpath("shared_ko_level_counts.tsv")
level_counts.write_csv(LEVEL_COUNTS_PATH, separator="\t")
with pl.Config(tbl_rows=20, tbl_cols=20, tbl_width_chars=400, fmt_str_lengths=30, ascii_tables=True):
    print(level_counts.drop("rule_label", "matched_level"))

Under the primary rule CheckAMG (high) holds 1,751 geNomad-viral calls with a KEGG Metabolism KO, 1,319 with a KO outside Metabolism, 83 with no KO and 68 calls off geNomad-viral regions. DRAM-V (aux 1) holds 1,637, 870, 628 and 245, and VIBRANT (high) 1,161, 98, 4 and 7. The levels are cumulative. At the second matched level CheckAMG (medium) holds 3,475, 2,216, 152 and 885, DRAM-V (aux 2) holds 5,404, 2,592, 1,494 and 4,076, and VIBRANT (medium) 3,442, 259, 7 and 12. At the third level CheckAMG (low) holds 5,456, 3,240, 183 and 161,282, DRAM-V (aux 3), which is DRAM-V (default), 10,244, 4,400, 2,102 and 30,562, and VIBRANT (low) 11,599, 885, 47 and 146. The sensitivity rule leaves the not-geNomad-viral counts unchanged and moves calls from both KO classes to no KO, so CheckAMG (high) becomes 841, 656, 1,656 and 68 and DRAM-V (aux 1) becomes 723, 237, 2,175 and 245. Of the 1,319 CheckAMG (high) viral calls with a KO outside Metabolism under the primary rule, 997 are Class II and 322 other KO. The 72 count cells of the level 1 rows are unchanged from the earlier disjoint table, and the cumulative level 2 and level 3 rows have no counterpart there.

## Outputs